# 🫀 Heart Disease Risk Prediction 2026

Notebook ini berisi pipeline lengkap prediksi risiko penyakit jantung menggunakan dataset `heart_disease_risk_2026.csv`.

**Langkah-langkah:**
1. Import Library
2. Load & Eksplorasi Data (EDA)
3. Preprocessing (Encoding, Scaling, Split)
4. Training 3 Model (Logistic Regression, Random Forest, XGBoost)
5. Evaluasi & Perbandingan Model
6. Feature Importance
7. Inference – Prediksi Data Pasien Baru

## 1. 📦 Import Library

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data Manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisasi ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# ── Model ─────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Evaluasi ──────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

# ── Config Visual ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Semua library berhasil di-import!')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')

## 2. 📂 Load & Eksplorasi Data (EDA)

In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('heart_disease_risk_2026.csv')

# Hapus kolom ID karena tidak relevan untuk prediksi
df.drop(columns=['patient_id'], inplace=True)

print('=' * 60)
print('📊 INFORMASI DATASET')
print('=' * 60)
print(f'Jumlah baris   : {df.shape[0]:,}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'Missing values : {df.isnull().sum().sum()}')
print()
df.head()

In [ ]:
# ── Tipe Data & Info ──────────────────────────────────────────────────────────
print('📋 INFO KOLOM')
print('-' * 60)
df.info()

In [ ]:
# ── Statistik Deskriptif ───────────────────────────────────────────────────────
print('📈 STATISTIK DESKRIPTIF (Numerik)')
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# ── Distribusi Target ─────────────────────────────────────────────────────────
target_counts = df['has_heart_disease'].value_counts()
target_pct    = df['has_heart_disease'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribusi Target: has_heart_disease', fontsize=15, fontweight='bold')

colors = ['#4FC3F7', '#EF5350']
labels = ['Tidak Sakit (0)', 'Sakit Jantung (1)']

# Bar chart
axes[0].bar(labels, target_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, (v, p) in enumerate(zip(target_counts.values, target_pct.values)):
    axes[0].text(i, v + 50, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Jumlah Data per Kelas')
axes[0].set_ylabel('Jumlah')
axes[0].set_ylim(0, target_counts.max() * 1.15)

# Pie chart
axes[1].pie(
    target_counts.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('Proporsi Kelas')

plt.tight_layout()
plt.show()

print(f'Rasio Imbalance: {target_pct[0]:.1f}% vs {target_pct[1]:.1f}%')

In [ ]:
# ── Distribusi Fitur Numerik ───────────────────────────────────────────────────
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
num_cols = [c for c in num_cols if c != 'has_heart_disease']

n_cols = 4
n_rows = (len(num_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
fig.suptitle('Distribusi Fitur Numerik', fontsize=16, fontweight='bold', y=1.01)
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(
        data=df, x=col, hue='has_heart_disease',
        kde=True, ax=axes[i], palette={0: '#4FC3F7', 1: '#EF5350'},
        alpha=0.6, bins=30
    )
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribusi Fitur Kategorikal ───────────────────────────────────────────────
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f'Kolom Kategorikal: {cat_cols}')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribusi Fitur Kategorikal vs Target', fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['has_heart_disease'], normalize='index') * 100
    ct.plot(
        kind='bar', ax=axes[i], color=['#4FC3F7', '#EF5350'],
        edgecolor='white', linewidth=0.5
    )
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Persentase (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(['Tidak Sakit', 'Sakit Jantung'], fontsize=8)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap Korelasi ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 12))
corr = df.select_dtypes(include=['float64', 'int64']).corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, ax=ax,
    annot_kws={'size': 8}, linewidths=0.5
)
ax.set_title('Heatmap Korelasi Antar Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. 🔧 Preprocessing

In [ ]:
# ── Cek Missing Values ────────────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing Values per Kolom:')
print(missing[missing > 0] if missing.any() else '✅ Tidak ada missing values!')

In [ ]:
# ── Encoding Fitur Kategorikal ────────────────────────────────────────────────
df_processed = df.copy()

# Boolean columns → int
bool_cols = ['exercise_induced_angina', 'family_history', 'wearable_owner']
for col in bool_cols:
    if df_processed[col].dtype == object:
        df_processed[col] = df_processed[col].map({'True': 1, 'False': 0})
    else:
        df_processed[col] = df_processed[col].astype(int)

# Label Encoding untuk kolom biner
le = LabelEncoder()
df_processed['sex'] = le.fit_transform(df_processed['sex'])  # Male=1, Female=0

# One-Hot Encoding untuk kolom multi-kategori
df_processed = pd.get_dummies(
    df_processed,
    columns=['chest_pain_type', 'smoker_status'],
    drop_first=True
)

print(f'Jumlah fitur setelah encoding: {df_processed.shape[1] - 1}')
print(f'Kolom baru: {df_processed.columns.tolist()}')

In [ ]:
# ── Pisahkan Fitur & Target ───────────────────────────────────────────────────
X = df_processed.drop(columns=['has_heart_disease'])
y = df_processed['has_heart_disease']

print(f'Shape X : {X.shape}')
print(f'Shape y : {y.shape}')
print(f'Distribusi y : {dict(y.value_counts())}')

In [ ]:
# ── Train-Test Split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Menjaga proporsi kelas
)

print('=' * 50)
print('📊 HASIL SPLIT DATA')
print('=' * 50)
print(f'Training set  : {X_train.shape[0]:,} sampel ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Testing set   : {X_test.shape[0]:,} sampel ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Jumlah fitur  : {X_train.shape[1]}')

In [ ]:
# ── Feature Scaling (untuk Logistic Regression) ───────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Feature scaling selesai!')
print(f'   Mean (train) ≈ {X_train_scaled.mean():.4f}  (harusnya ≈ 0)')
print(f'   Std  (train) ≈ {X_train_scaled.std():.4f}   (harusnya ≈ 1)')

## 4. 🤖 Training Model

Tiga model yang akan digunakan:
| Model | Keterangan |
|---|---|
| **Logistic Regression** | Model linear, mudah diinterpretasi |
| **Random Forest** | Ensemble pohon keputusan, robust |
| **XGBoost** | Gradient boosting, performa tinggi |

In [ ]:
# ── Definisi Model ────────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver='lbfgs',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
}

print(f'✅ {len(models)} model siap dilatih.')

In [ ]:
# ── Training & Cross-Validation ───────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('🚀 Mulai training...')
print('=' * 60)

for name, model in models.items():
    print(f'\n⏳ Training: {name}')

    # Gunakan data scaled untuk LR, data asli untuk tree-based
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train
    X_te = X_test_scaled  if name == 'Logistic Regression' else X_test

    # Fit model
    model.fit(X_tr, y_train)

    # Prediksi
    y_pred      = model.predict(X_te)
    y_pred_prob = model.predict_proba(X_te)[:, 1]

    # Cross-validation score
    cv_scores = cross_val_score(
        model, X_tr, y_train,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )

    # Simpan hasil
    results[name] = {
        'model'       : model,
        'y_pred'      : y_pred,
        'y_pred_prob' : y_pred_prob,
        'accuracy'    : accuracy_score(y_test, y_pred),
        'precision'   : precision_score(y_test, y_pred, zero_division=0),
        'recall'      : recall_score(y_test, y_pred, zero_division=0),
        'f1'          : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'     : roc_auc_score(y_test, y_pred_prob),
        'cv_mean'     : cv_scores.mean(),
        'cv_std'      : cv_scores.std()
    }

    print(f'   ✅ Selesai | Accuracy: {results[name]["accuracy"]:.4f} | ROC-AUC: {results[name]["roc_auc"]:.4f} | CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\n' + '=' * 60)
print('🎉 Semua model selesai dilatih!')

## 5. 📊 Evaluasi Model

In [ ]:
# ── Tabel Perbandingan Metrik ─────────────────────────────────────────────────
metrics_df = pd.DataFrame([
    {
        'Model'      : name,
        'Accuracy'   : f"{r['accuracy']:.4f}",
        'Precision'  : f"{r['precision']:.4f}",
        'Recall'     : f"{r['recall']:.4f}",
        'F1-Score'   : f"{r['f1']:.4f}",
        'ROC-AUC'    : f"{r['roc_auc']:.4f}",
        'CV AUC'     : f"{r['cv_mean']:.4f} ± {r['cv_std']:.4f}"
    }
    for name, r in results.items()
]).set_index('Model')

print('📋 PERBANDINGAN METRIK MODEL')
print('=' * 80)
metrics_df

In [ ]:
# ── Visualisasi: Bar Chart Metrik ─────────────────────────────────────────────
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names = list(results.keys())
colors_model = ['#42A5F5', '#66BB6A', '#FFA726']

x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, color) in enumerate(zip(model_names, colors_model)):
    vals = [results[name][m] for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color, edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold'
        )

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title('Perbandingan Metrik Evaluasi Antar Model', fontsize=14, fontweight='bold')
ax.set_ylabel('Skor')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrix', fontsize=15, fontweight='bold')

for ax, (name, r), color in zip(axes, results.items(), ['Blues', 'Greens', 'Oranges']):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap=color,
        xticklabels=['Pred: Tidak Sakit', 'Pred: Sakit'],
        yticklabels=['True: Tidak Sakit', 'True: Sakit'],
        ax=ax, linewidths=0.5, linecolor='white',
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(
        f'{name}\nAccuracy: {results[name]["accuracy"]:.3f}',
        fontsize=11, fontweight='bold'
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#42A5F5', '#66BB6A', '#FFA726']

for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_pred_prob'])
    ax.plot(
        fpr, tpr, lw=2.5, color=color,
        label=f"{name} (AUC = {r['roc_auc']:.4f})"
    )

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.5000)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve – Perbandingan Model', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.4)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

In [ ]:
# ── Classification Report ─────────────────────────────────────────────────────
for name, r in results.items():
    print('=' * 60)
    print(f'📋 Classification Report: {name}')
    print('=' * 60)
    print(classification_report(
        y_test, r['y_pred'],
        target_names=['Tidak Sakit', 'Sakit Jantung']
    ))
    print()

## 6. 🏆 Feature Importance

In [ ]:
# ── Feature Importance – Random Forest ───────────────────────────────────────
rf_model = results['Random Forest']['model']
rf_importance = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

top_n = 15
fig, ax = plt.subplots(figsize=(11, 7))

colors_imp = sns.color_palette('viridis', top_n)
bars = ax.barh(
    rf_importance['Feature'][:top_n][::-1],
    rf_importance['Importance'][:top_n][::-1],
    color=colors_imp[::-1], edgecolor='white'
)

for bar, v in zip(bars, rf_importance['Importance'][:top_n][::-1]):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
            f'{v:.4f}', va='center', fontsize=9)

ax.set_title(f'Top {top_n} Feature Importance – Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance – XGBoost ─────────────────────────────────────────────
xgb_model = results['XGBoost']['model']
xgb_importance = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 7))
colors_xgb = sns.color_palette('plasma', top_n)

bars = ax.barh(
    xgb_importance['Feature'][:top_n][::-1],
    xgb_importance['Importance'][:top_n][::-1],
    color=colors_xgb[::-1], edgecolor='white'
)

for bar, v in zip(bars, xgb_importance['Importance'][:top_n][::-1]):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
            f'{v:.4f}', va='center', fontsize=9)

ax.set_title(f'Top {top_n} Feature Importance – XGBoost', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── Ringkasan Final ───────────────────────────────────────────────────────────
best_model = max(results, key=lambda n: results[n]['roc_auc'])

print('=' * 60)
print('🏆 RINGKASAN EVALUASI MODEL')
print('=' * 60)
for name, r in results.items():
    marker = '👑' if name == best_model else '  '
    print(f"{marker} {name:<25} | ROC-AUC: {r['roc_auc']:.4f} | F1: {r['f1']:.4f} | Acc: {r['accuracy']:.4f}")

print('=' * 60)
print(f'🥇 Model Terbaik (ROC-AUC): {best_model}')
print(f'   ROC-AUC  : {results[best_model]["roc_auc"]:.4f}')
print(f'   Accuracy : {results[best_model]["accuracy"]:.4f}')
print(f'   F1-Score : {results[best_model]["f1"]:.4f}')
print('=' * 60)

## 7. 🔬 Inference – Prediksi dari Data Pasien Baru

Masukkan data pasien baru pada dictionary `data_pasien_baru` di cell berikutnya,
lalu jalankan semua cell di bawah untuk mendapatkan prediksi dari ketiga model.

### Panduan Nilai yang Valid
| Kolom | Tipe | Contoh / Nilai Valid |
|---|---|---|
| `age` | int | `45` |
| `sex` | str | `'Male'` atau `'Female'` |
| `resting_bp_systolic` | int | `120` |
| `resting_bp_diastolic` | int | `80` |
| `cholesterol_total` | int | `200` |
| `hdl` | int | `55` |
| `ldl` | int | `110` |
| `triglycerides` | int | `150` |
| `fasting_blood_sugar` | int | `100` |
| `hba1c` | float | `5.5` |
| `bmi` | float | `24.0` |
| `resting_heart_rate` | int | `72` |
| `max_heart_rate_achieved` | int | `160` |
| `chest_pain_type` | str | `'Asymptomatic'` / `'Atypical Angina'` / `'Non-Anginal Pain'` / `'Typical Angina'` |
| `exercise_induced_angina` | bool | `True` atau `False` |
| `st_depression` | float | `1.0` |
| `family_history` | bool | `True` atau `False` |
| `smoker_status` | str | `'Never'` / `'Former'` / `'Current'` |
| `alcohol_units_per_week` | float | `3.0` |
| `exercise_minutes_per_week` | int | `150` |
| `sleep_hours` | float | `7.0` |
| `stress_score` | float | `40.0` |
| `wearable_owner` | bool | `True` atau `False` |
| `daily_steps` | int | `8000` |
| `diet_quality_score` | float | `60.0` |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         ✏️  EDIT DATA PASIEN BARU DI SINI                   ║
# ╚══════════════════════════════════════════════════════════════╝

data_pasien_baru = {
    # ── Demografis ──────────────────────────────────────────────
    'age'                       : 55,
    'sex'                       : 'Male',          # 'Male' / 'Female'

    # ── Tekanan Darah ────────────────────────────────────────────
    'resting_bp_systolic'       : 140,
    'resting_bp_diastolic'      : 90,

    # ── Profil Lipid & Gula Darah ────────────────────────────────
    'cholesterol_total'         : 230,
    'hdl'                       : 45,
    'ldl'                       : 150,
    'triglycerides'             : 180,
    'fasting_blood_sugar'       : 115,
    'hba1c'                     : 6.2,

    # ── Komposisi Tubuh & Kardio ──────────────────────────────────
    'bmi'                       : 28.5,
    'resting_heart_rate'        : 78,
    'max_heart_rate_achieved'   : 145,

    # ── Gejala Klinis ─────────────────────────────────────────────
    'chest_pain_type'           : 'Asymptomatic',  # 'Asymptomatic'/'Atypical Angina'/'Non-Anginal Pain'/'Typical Angina'
    'exercise_induced_angina'   : True,             # True / False
    'st_depression'             : 1.5,

    # ── Riwayat & Gaya Hidup ──────────────────────────────────────
    'family_history'            : True,             # True / False
    'smoker_status'             : 'Former',         # 'Never' / 'Former' / 'Current'
    'alcohol_units_per_week'    : 5.0,
    'exercise_minutes_per_week' : 60,
    'sleep_hours'               : 6.0,
    'stress_score'              : 55.0,
    'wearable_owner'            : False,            # True / False
    'daily_steps'               : 5000,
    'diet_quality_score'        : 45.0,
}

print('✅ Data pasien baru siap diproses.')
pd.DataFrame([data_pasien_baru]).T.rename(columns={0: 'Nilai'}).style.set_caption('📋 Data Pasien Baru')

In [ ]:
# ── Fungsi Preprocessing untuk Data Inferensi ─────────────────────────────────
def preprocess_input(raw_data: dict, feature_columns: list, fitted_scaler) -> tuple:
    """
    Mengubah raw dict pasien menjadi DataFrame siap pakai model.
    Returns: (df_raw, df_scaled)
      - df_raw    : untuk tree-based model (RF, XGBoost)
      - df_scaled : untuk Logistic Regression
    """
    df_in = pd.DataFrame([raw_data])

    # 1. Boolean → int
    bool_cols_inf = ['exercise_induced_angina', 'family_history', 'wearable_owner']
    for col in bool_cols_inf:
        if col in df_in.columns:
            df_in[col] = df_in[col].astype(int)

    # 2. Sex → int (Male=1, Female=0)
    df_in['sex'] = df_in['sex'].map({'Male': 1, 'Female': 0})

    # 3. One-Hot Encoding (konsisten dengan training: drop_first=True)
    df_in = pd.get_dummies(df_in, columns=['chest_pain_type', 'smoker_status'], drop_first=True)

    # 4. Tambahkan kolom OHE yang mungkin tidak muncul di input ini (isi 0)
    for col in feature_columns:
        if col not in df_in.columns:
            df_in[col] = 0

    # 5. Urutkan kolom sesuai urutan training
    df_in = df_in[feature_columns]

    # 6. Scaled version untuk Logistic Regression
    df_scaled = fitted_scaler.transform(df_in)

    return df_in, df_scaled


# ── Jalankan Preprocessing ────────────────────────────────────────────────────
feature_columns  = list(X.columns)
df_input, df_input_scaled = preprocess_input(data_pasien_baru, feature_columns, scaler)

print('✅ Preprocessing selesai!')
print(f'   Shape input : {df_input.shape}')
print(f'   Kolom aktif : {df_input.columns.tolist()}')

In [ ]:
# ── Prediksi dari Semua Model ─────────────────────────────────────────────────
label_map    = {0: '✅ Tidak Berisiko', 1: '⚠️  Berisiko Sakit Jantung'}
risk_color   = {0: '\033[92m', 1: '\033[91m'}   # hijau / merah ANSI
RESET        = '\033[0m'

inference_results = {}

print('=' * 65)
print('🔬 HASIL PREDIKSI UNTUK PASIEN BARU')
print('=' * 65)

for name, r in results.items():
    model_obj = r['model']

    # Pilih format input yang sesuai
    X_inf = df_input_scaled if name == 'Logistic Regression' else df_input

    pred      = model_obj.predict(X_inf)[0]
    prob_risk = model_obj.predict_proba(X_inf)[0][1]   # probabilitas kelas 1

    inference_results[name] = {'pred': int(pred), 'prob': float(prob_risk)}

    c = risk_color[int(pred)]
    print(f'\n  📌 {name}')
    print(f'     Prediksi           : {c}{label_map[int(pred)]}{RESET}')
    print(f'     Prob. Risiko (%)   : {prob_risk*100:.2f}%')
    print(f'     Prob. Aman   (%)   : {(1-prob_risk)*100:.2f}%')

print('\n' + '=' * 65)

# ── Voting Mayoritas (2 dari 3 model setuju) ───────────────────────────────────
votes      = [v['pred'] for v in inference_results.values()]
final_vote = 1 if votes.count(1) >= 2 else 0
avg_prob   = sum(v['prob'] for v in inference_results.values()) / len(inference_results)

c_final = risk_color[final_vote]
print(f'\n  🗳️  Voting Mayoritas  : {c_final}{label_map[final_vote]}{RESET}')
print(f'     Rata-rata Prob.   : {avg_prob*100:.2f}%')
print('=' * 65)

In [ ]:
# ── Visualisasi Probabilitas Risiko ───────────────────────────────────────────
model_names_inf = list(inference_results.keys())
probs           = [inference_results[n]['prob'] * 100 for n in model_names_inf]
preds           = [inference_results[n]['pred'] for n in model_names_inf]
bar_colors      = ['#EF5350' if p == 1 else '#66BB6A' for p in preds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🔬 Hasil Inference – Probabilitas Risiko Penyakit Jantung',
             fontsize=14, fontweight='bold')

# ── Panel Kiri: Bar Chart per Model ──────────────────────────────────────────
bars = axes[0].bar(model_names_inf, probs,
                   color=bar_colors, edgecolor='white', linewidth=1.5, width=0.45)
axes[0].axhline(50, color='gray', linestyle='--', lw=1.5, label='Threshold 50%')

for bar, v, pred in zip(bars, probs, preds):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f'{v:.1f}%\n({"⚠️ Risiko" if pred == 1 else "✅ Aman"})',
        ha='center', fontsize=9, fontweight='bold'
    )

axes[0].set_ylim(0, 118)
axes[0].set_ylabel('Probabilitas Risiko (%)')
axes[0].set_title('Probabilitas per Model')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.4)
axes[0].tick_params(axis='x', labelsize=9)

# ── Panel Kanan: Gauge Chart (rata-rata probabilitas) ────────────────────────
theta      = np.linspace(0, np.pi, 300)
r_out, r_in = 1.0, 0.55
gauge_col  = '#EF5350' if avg_prob >= 0.5 else '#66BB6A'

# Latar belakang abu-abu
axes[1].fill_between(
    np.cos(theta) * r_out, np.sin(theta) * r_out,
    np.cos(theta) * r_in,  np.sin(theta) * r_in,
    color='#ECEFF1', zorder=1
)

# Arc terisi sesuai probabilitas
theta_fill = np.linspace(0, np.pi * avg_prob, 300)
axes[1].fill_between(
    np.cos(theta_fill) * r_out, np.sin(theta_fill) * r_out,
    np.cos(theta_fill) * r_in,  np.sin(theta_fill) * r_in,
    color=gauge_col, zorder=2, alpha=0.85
)

# Jarum penunjuk
needle_angle = np.pi * avg_prob
axes[1].annotate(
    '', xy=(np.cos(needle_angle) * 0.75, np.sin(needle_angle) * 0.75),
    xytext=(0, 0),
    arrowprops=dict(arrowstyle='->', color='#212121', lw=2.5)
)
axes[1].plot(0, 0, 'o', color='#212121', ms=8, zorder=5)

# Label teks
axes[1].text(0, -0.18, f'{avg_prob*100:.1f}%',
             ha='center', fontsize=24, fontweight='bold', color=gauge_col)
verdict = label_map[final_vote].replace('✅ ', '').replace('⚠️  ', '')
axes[1].text(0, -0.36, verdict,
             ha='center', fontsize=11, color=gauge_col, fontweight='bold')
axes[1].text(-1.08, -0.06, '0%',   fontsize=9, ha='center', color='#90A4AE')
axes[1].text( 1.08, -0.06, '100%', fontsize=9, ha='center', color='#90A4AE')
axes[1].text( 0,    1.12,  '50%',  fontsize=9, ha='center', color='#90A4AE')

axes[1].set_xlim(-1.35, 1.35)
axes[1].set_ylim(-0.5, 1.35)
axes[1].set_aspect('equal')
axes[1].axis('off')
axes[1].set_title('Gauge – Rata-rata Probabilitas (3 Model)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Tabel Ringkasan Inference ─────────────────────────────────────────────────
rows = [
    {
        'Model'              : name,
        'Prediksi'           : '⚠️ Risiko' if v['pred'] == 1 else '✅ Aman',
        'Prob. Risiko (%)'   : f"{v['prob']*100:.2f}%",
        'Prob. Aman   (%)'   : f"{(1 - v['prob'])*100:.2f}%"
    }
    for name, v in inference_results.items()
]
rows.append({
    'Model'              : '🗳️ Voting Mayoritas',
    'Prediksi'           : '⚠️ Risiko' if final_vote == 1 else '✅ Aman',
    'Prob. Risiko (%)'   : f'{avg_prob*100:.2f}%',
    'Prob. Aman   (%)'   : f'{(1-avg_prob)*100:.2f}%'
})

inf_summary = pd.DataFrame(rows).set_index('Model')

print('=' * 65)
print('📋 RINGKASAN INFERENCE PASIEN BARU')
print('=' * 65)
print(inf_summary.to_string())
print('=' * 65)
print()
print('💡 Catatan: Prediksi ini bersifat INDIKATIF dan bukan pengganti')
print('   diagnosis medis dari tenaga kesehatan profesional.')
print('=' * 65)